# 🔬 Lab W6-3 — เมื่อใดควรทิ้งผลการแบ่งกลุ่ม

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 6 — Data Mining II**

Lab นี้ใช้คู่กับสื่อจำลอง **Cluster Reality Check** (`/sims/cluster-reality-check`)
ค่า silhouette และการเลือก k ในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง
(ดูหมายเหตุเรื่อง inertia ท้ายเล่ม)

## สิ่งที่จะได้เรียนรู้
1. พิสูจน์ว่า **K-Means คืนกลุ่มมาให้เสมอ** แม้ข้อมูลไม่มีโครงสร้างเลย
2. ใช้ **ข้อมูลอ้างอิงแบบสุ่ม** เป็นเกณฑ์เทียบก่อนเชื่อผลการแบ่งกลุ่ม
3. วัด **ความเสถียรของกลุ่ม** ด้วย Adjusted Rand Index
4. เขียน **เกณฑ์การยอมรับ** ที่ทีมใช้ได้จริงก่อนนำผลไปเสนอผู้บริหาร

## ข้อมูล
* `customers_rfm.csv` — ลูกค้าจริง 1,200 ราย ที่มีกลุ่มพฤติกรรมอยู่จริง
* `no_structure.csv` — จุดสุ่มสม่ำเสมอ 900 จุด **ที่ไม่มีกลุ่มอยู่จริงเลย**

ทั้งสองไฟล์มีชื่อคอลัมน์เหมือนกันทุกประการ จึงสลับใช้ได้โดยไม่ต้องแก้โค้ดแม้แต่บรรทัดเดียว

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

BASE = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
        "master/datasets/week06/")
F = ["recency_days", "frequency", "monetary"]

real = pd.read_csv(BASE + "customers_rfm.csv").sort_values("customer_id").reset_index(drop=True)
noise = pd.read_csv(BASE + "no_structure.csv").sort_values("customer_id").reset_index(drop=True)

X_real = StandardScaler().fit_transform(real[F].astype(float))
X_noise = StandardScaler().fit_transform(noise[F].astype(float))

print(f"ข้อมูลลูกค้าจริง : {X_real.shape}")
print(f"จุดสุ่มล้วน      : {X_noise.shape}")
print("\nทั้งสองชุดผ่านการปรับสเกลแบบเดียวกัน และจะถูกวิเคราะห์ด้วยโค้ดบรรทัดเดียวกัน")

## ส่วนที่ 1 — รันแบบเดียวกันกับทั้งสองชุด

### 🧑‍💻 งานที่ 1
เขียนฟังก์ชัน `sweep(X)` ที่คืน `DataFrame` ของ inertia และ silhouette
สำหรับ k = 2 ถึง 8 แล้วรันกับทั้งสองชุดข้อมูล

**อย่าเพิ่งดูว่าชุดไหนเป็นชุดไหน** — ลองอ่านตัวเลขก่อนแล้วเดาว่าชุดใดมีโครงสร้างจริง

*เฉลยที่ถูกต้อง: ข้อมูลจริง silhouette สูงสุด 0.5847 ที่ k = 4
ส่วนจุดสุ่มได้ 0.2868 ที่ k = 6*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 2 — สัญญาณสี่ข้อที่แยกทั้งสองชุดออกจากกัน

### 🧑‍💻 งานที่ 2
สร้างตารางเปรียบเทียบสัญญาณต่อไปนี้ระหว่างสองชุด

1. ค่า silhouette สูงสุด
2. ช่วงห่างระหว่าง silhouette สูงสุดกับต่ำสุด
3. อัตราส่วนระหว่างกลุ่มใหญ่สุดกับกลุ่มเล็กสุด ที่ k ที่ดีที่สุด
4. เส้น silhouette มี "ยอด" ที่ชัดเจนหรือไม่

แล้วอธิบายว่าเหตุใดสัญญาณข้อ 2 และ 3 จึงบอกได้มากกว่าข้อ 1

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — ความเสถียรของกลุ่ม

ถ้ากลุ่มมีอยู่จริง การสุ่มตัวอย่างมาแค่บางส่วนก็ยังควรพบกลุ่มเดิม

### 🧑‍💻 งานที่ 3
สุ่มข้อมูลย่อย 80% จำนวน 10 รอบ (ใช้ `random_state` ต่างกันทุกรอบ)
รัน K-Means ที่ k ที่ดีที่สุดของแต่ละชุด แล้ววัดว่าการจัดกลุ่มของแต่ละรอบ
ตรงกับการจัดกลุ่มของโมเดลที่ฝึกด้วยข้อมูลเต็มมากแค่ไหน ด้วย **Adjusted Rand Index**

ARI = 1 คือเหมือนกันทุกประการ · ARI ≈ 0 คือตรงกันแค่ระดับบังเอิญ

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 4 — K-Means บอกว่า "ไม่มีกลุ่ม" ไม่ได้

### 🧑‍💻 งานที่ 4
ลองรัน DBSCAN กับทั้งสองชุด แล้วเทียบว่ามันตอบต่างจาก K-Means อย่างไร

แล้วตอบว่าเหตุใด DBSCAN จึงบอกได้ว่า "ไม่มีกลุ่ม" ในขณะที่ K-Means บอกไม่ได้

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 5 — เกณฑ์การยอมรับ

### 🧑‍💻 งานที่ 5
เขียนฟังก์ชัน `should_i_trust(X, k)` ที่ตรวจสัญญาณทั้งหมดที่เรียนมา
แล้วคืนคำตอบว่า "ยอมรับ" หรือ "ทิ้ง" พร้อมเหตุผลรายข้อ

ทดสอบกับทั้งสองชุด แล้วยืนยันว่าฟังก์ชันตอบถูกทั้งคู่

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 6 — เขียนเป็นนโยบายของทีม

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. เขียนเกณฑ์การยอมรับผลการแบ่งกลุ่มเป็นรายการที่ทีมคุณจะใช้กับทุกโปรเจกต์
   โดยต้องมีทั้งเกณฑ์เชิงสถิติและเกณฑ์เชิงธุรกิจ
2. ถ้าผลไม่ผ่านเกณฑ์ แต่ผู้บริหารกำลังรอผลอยู่ คุณจะสื่อสารอย่างไร
   เขียนเป็นข้อความจริงที่จะส่งไป ไม่เกิน 5 บรรทัด
3. ยกตัวอย่างสถานการณ์ที่ "ไม่มีกลุ่ม" เป็นคำตอบที่ **มีประโยชน์** ต่อธุรกิจ

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — sweep ทั้งสองชุดและได้ตัวเลขตรงเฉลย | 3 |
| งานที่ 2 — เปรียบเทียบสัญญาณ 4 ข้อพร้อมคำอธิบาย | 4 |
| งานที่ 3 — วัดความเสถียรด้วย ARI | 3 |
| งานที่ 4 — เทียบกับ DBSCAN และอธิบายความต่างเชิงหลักการ | 3 |
| งานที่ 5 — ฟังก์ชันเกณฑ์การยอมรับที่ตัดสินถูกทั้งสองชุด | 4 |
| งานที่ 6 — นโยบายของทีมและการสื่อสารกับผู้บริหาร | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/cluster-reality-check`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง

> 💡 **หมายเหตุเรื่องการเทียบตัวเลขกับสื่อจำลอง**
> สื่อจำลองใช้ K-Means ที่กำหนดค่าเริ่มต้นของ centroid แบบตายตัว ส่วน Lab นี้ใช้
> `KMeans` ของ scikit-learn ที่ใช้ k-means++ และรัน 10 รอบเลือกผลที่ดีที่สุด
>
> **ค่าที่ต้องตรงกัน:** silhouette ของแต่ละ k · ค่า k ที่ดีที่สุด · ขนาดและโปรไฟล์ของแต่ละกลุ่ม
> **ค่าที่อาจต่างในทศนิยมท้าย ๆ:** inertia — เพราะขึ้นกับค่าเริ่มต้น
>
> ความต่างนี้เองเป็นบทเรียน: **K-Means ไวต่อค่าเริ่มต้น** จึงต้องตั้ง `n_init` และ
> `random_state` เสมอ มิฉะนั้นผลจะไม่ซ้ำเดิมแม้รันบนข้อมูลชุดเดียวกัน